# Aula 02: O Problema do Caixeiro Viajante (TSP) via Programação Linear Inteira

Nesta aula, focaremos na modelagem do TSP como um problema de **Programação Linear Inteira (PLI)**. Como estamos no módulo de Programação Linear, entenderemos como converter um problema combinatório complexo em um conjunto de equações lineares.

## 1. O Desafio da Modelagem em PLI

O TSP busca o ciclo Hamiltoniano de custo mínimo. Simplesmente garantir que cada cidade tenha uma entrada e uma saída não é suficiente, pois isso permitiria a formação de **sub-ciclos** (caminhos fechados que não visitam todas as cidades).

### Variáveis de Decisão
- $x_{ij} = 1$ se o caixeiro viaja da cidade $i$ diretamente para a cidade $j$.
- $x_{ij} = 0$ caso contrário.

## 2. Formulação MTZ (Miller-Tucker-Zemlin)

A formulação MTZ é uma das formas mais comuns de evitar sub-ciclos em um modelo de PLI usando um número polinomial de restrições.

### Modelo Matemático

**Função Objetivo:**
Minimizar $\sum_{i=1}^n \sum_{j=1, j \neq i}^n c_{ij} x_{ij}$

**Restrições de Grau (Atribuição):**
1. Cada cidade deve ser deixada exatamente uma vez:
   $\sum_{j=1, j \neq i}^n x_{ij} = 1, \quad \forall i \in \{1, \dots, n\}$

2. Cada cidade deve ser visitada exatamente uma vez:
   $\sum_{i=1, i \neq j}^n x_{ij} = 1, \quad \forall j \in \{1, \dots, n\}$

**Restrições de Eliminação de Sub-ciclos (MTZ):**
Para cidades $i, j \in \{2, \dots, n\}$ e $i \neq j$:
$u_i - u_j + n x_{ij} \leq n - 1$

Onde $u_i$ são variáveis auxiliares contínuas que representam a ordem de visita das cidades (exceto a cidade de origem, geralmente a 1).

## 3. Implementação e Visualização com PuLP

Vamos implementar o modelo MTZ para resolver o TSP de forma exata e gerar uma visualização gráfica da rota.

In [ ]:
import pulp
import math
import matplotlib.pyplot as plt

# Definição das cidades (exemplo simplificado)
cidades = [
    {"id": 0, "x": 0, "y": 0},
    {"id": 1, "x": 4, "y": 0},
    {"id": 2, "x": 0, "y": 3},
    {"id": 3, "x": 4, "y": 3}
]
n = len(cidades)

def dist(i, j, lista_cidades):
    c1, c2 = lista_cidades[i], lista_cidades[j]
    return math.sqrt((c1['x'] - c2['x'])**2 + (c1['y'] - c2['y'])**2)

# 1. Inicializar o modelo
prob = pulp.LpProblem("TSP_MTZ", pulp.LpMinimize)

# 2. Variáveis de Decisão
x = pulp.LpVariable.dicts("x", (range(n), range(n)), cat='Binary')
u = pulp.LpVariable.dicts("u", range(n), lowBound=1, upBound=n, cat='Continuous')

# 3. Função Objetivo
prob += pulp.lpSum(dist(i, j, cidades) * x[i][j] for i in range(n) for j in range(n) if i != j)

# 4. Restrições de Atribuição
for i in range(n):
    prob += pulp.lpSum(x[i][j] for j in range(n) if i != j) == 1
for j in range(n):
    prob += pulp.lpSum(x[i][j] for i in range(n) if i != j) == 1

# 5. Restrições MTZ (Eliminação de Sub-ciclos)
for i in range(1, n):
    for j in range(1, n):
        if i != j:
            prob += u[i] - u[j] + n * x[i][j] <= n - 1

# 6. Resolver
status = prob.solve(pulp.PULP_CBC_CMD(msg=0))

# 7. Função para Visualização
def plot_tour(lista_cidades, x_vars):
    plt.figure(figsize=(8, 6))
    xs = [c['x'] for c in lista_cidades]
    ys = [c['y'] for c in lista_cidades]
    plt.scatter(xs, ys, color='red', zorder=5)
    
    for i, c in enumerate(lista_cidades):
        plt.annotate(c['id'], (c['x'], c['y']), textcoords="offset points", xytext=(0,10), ha='center')
    
    for i in range(len(lista_cidades)):
        for j in range(len(lista_cidades)):
            if i != j and pulp.value(x_vars[i][j]) == 1:
                plt.plot([lista_cidades[i]['x'], lista_cidades[j]['x']], 
                         [lista_cidades[i]['y'], lista_cidades[j]['y']], 
                         color='blue', linestyle='--', alpha=0.6)
    
    plt.title(f"Visualização da Rota Ótima (Custo: {pulp.value(prob.objective):.2f})")
    plt.grid(True)
    plt.show()

print(f"Status: {pulp.LpStatus[status]}")
plot_tour(cidades, x)

## 4. DFJ vs MTZ

Existem duas formulações clássicas para o TSP:

1. **DFJ (Dantzig-Fulkerson-Johnson)**: Usa restrições que proíbem sub-ciclos em todos os subconjuntos possíveis de cidades. 
   - **Vantagem:** Muito forte matematicamente (limite inferior melhor).
   - **Desvantagem:** Existem $2^n$ subconjuntos, o que torna impossível listar todas as restrições para $n$ grande.

2. **MTZ (Miller-Tucker-Zemlin)**: 
   - **Vantagem:** Possui apenas $O(n^2)$ restrições, cabendo facilmente na memória do computador.
   - **Desvantagem:** É uma formulação mais "fraca" e demora mais para ser resolvida por solvers em problemas muito grandes.

## 5. Teste de Performance com Instância Maior

Agora que entendemos a teoria e a implementação, vamos testar o modelo MTZ com uma instância maior. Na pasta `data`, criamos um arquivo chamado `tsp_grande.json` com 15 cidades.

**DESAFIO**: Carregue os dados abaixo e adapte o código da seção 3 para resolver esta instância. **Tente também gerar a visualização da rota para as 15 cidades!**

In [ ]:
import json

# Carregar a instância grande
try:
    with open('data/tsp_grande.json', 'r') as f:
        data_grande = json.load(f)
        cidades_grande = data_grande['instancia_grande']['cidades']
        print(f"Sucesso! {len(cidades_grande)} cidades carregadas.")
except FileNotFoundError:
    print("Arquivo tsp_grande.json não encontrado na pasta data.")

# DICA: Ao recriar o modelo para 15 cidades, não esqueça de chamar a função plot_tour(cidades_grande, x)!

### Relatório de Observações

Após rodar a instância de 15 cidades, documente abaixo:
1. **Resultado**: Qual foi a distância mínima encontrada?
2. **Tempo de Execução**: Quanto tempo (aproximadamente) o solver levou?
3. **Observações Visuais**: A rota encontrada visualmente faz sentido? Há cruzamentos desnecessários?

## Exercício Proposto

1. Explique por que a restrição MTZ não se aplica à cidade de origem (cidade 0).
2. O que acontece se removermos as variáveis $u_i$ e as restrições de sub-ciclo? Tente comentar essa parte do código e rodar novamente. O resultado ainda será um ciclo único?